In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, Embedding, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy

# Toy example data
texts = ["Tashkent is a beautiful city","Tashkent retains a multiethnic population","It is the most populous city in Central Asia", "I didn't like the traffic."]
labels = [1,1,0, 0]  # 1: Positive, 0: Negative

# Tokenize texts
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
max_len = max(len(seq) for seq in sequences)
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')

# Define Transformer encoder layer
def transformer_encoder(inputs, d_model, num_heads, dff, dropout_rate):
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(inputs, inputs)
    attention_output = Dropout(dropout_rate)(attention_output)
    attention_output = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    ffn_output = Dense(dff, activation='relu')(attention_output)
    ffn_output = Dense(d_model)(ffn_output)
    ffn_output = Dropout(dropout_rate)(ffn_output)
    encoded_output = LayerNormalization(epsilon=1e-6)(attention_output + ffn_output)
    return encoded_output

# Define Transformer encoder
def transformer_encoder_stack(inputs, num_layers, d_model, num_heads, dff, dropout_rate):
    for _ in range(num_layers):
        inputs = transformer_encoder(inputs, d_model, num_heads, dff, dropout_rate)
    return inputs

# Define Transformer model
def transformer_model(input_shape, num_layers, d_model, num_heads, dff, dropout_rate):
    inputs = Input(shape=input_shape)
    encoded = Embedding(10000, d_model)(inputs)
    encoded = transformer_encoder_stack(encoded, num_layers, d_model, num_heads, dff, dropout_rate)
    pooled = GlobalAveragePooling1D()(encoded)
    outputs = Dense(2, activation='softmax')(pooled)
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Define hyperparameters
num_layers = 2
d_model = 64
num_heads = 4
dff = 128
dropout_rate = 0.1

# Instantiate and compile the model
model = transformer_model((max_len,), num_layers, d_model, num_heads, dff, dropout_rate)
model.compile(optimizer=Adam(), loss=SparseCategoricalCrossentropy(), metrics=[SparseCategoricalAccuracy()])

# Train the model
model.fit(padded_sequences, np.array(labels), epochs=10, batch_size=2)

# Test the model
test_texts = ["I loved the city"]
test_sequences = tokenizer.texts_to_sequences(test_texts)
padded_test_sequences = tf.keras.preprocessing.sequence.pad_sequences(test_sequences, maxlen=max_len, padding='post')
predictions = model.predict(padded_test_sequences)
print(predictions)


Epoch 1/10
2/2 [==============================] - 4s 18ms/step - loss: 1.2498 - sparse_categorical_accuracy: 0.5000
Epoch 2/10
2/2 [==============================] - 0s 23ms/step - loss: 0.5050 - sparse_categorical_accuracy: 0.7500
Epoch 3/10
2/2 [==============================] - 0s 23ms/step - loss: 0.2586 - sparse_categorical_accuracy: 1.0000
Epoch 4/10
2/2 [==============================] - 0s 17ms/step - loss: 0.1565 - sparse_categorical_accuracy: 1.0000
Epoch 5/10
2/2 [==============================] - 0s 21ms/step - loss: 0.0531 - sparse_categorical_accuracy: 1.0000
Epoch 6/10
2/2 [==============================] - 0s 19ms/step - loss: 0.0332 - sparse_categorical_accuracy: 1.0000
Epoch 7/10
2/2 [==============================] - 0s 22ms/step - loss: 0.0526 - sparse_categorical_accuracy: 1.0000
Epoch 8/10
2/2 [==============================] - 0s 18ms/step - loss: 0.0190 - sparse_categorical_accuracy: 1.0000
Epoch 9/10
2/2 [==============================] - 0s 20ms/step - loss: 0